# ML-02 — Research Question and Freestyle Lane

## 1. Lane and motivation

I choose Freestyle: Sibling-Page Search Interaction and Consolidation Review.

Same-site pages can share demand in several ways: one gains while a sibling declines, both grow
while demand remains fragmented, both decline with a topic-level problem, or both legitimately
serve complementary intent. The system discovers recurring page-pair patterns and never treats
overlap alone as harmful.

## 2. Decision, action, and error cost

Decision: Which same-client pairs should an editor review first for consolidation,
differentiation, protection, topic improvement, or monitoring?

The output is a ranked public-safe queue with an observed pattern and reason codes. A human checks
intent and business purpose first. False positives can destroy useful coverage; false negatives
can leave fragmented demand unresolved. Unsupervised learning fits because no verified
cannibalization label exists.

## 3. Research question

What recurring search-interaction patterns appear among same-client pages sharing visible query
demand, and how can they prioritize consolidation and differentiation review without treating
overlap as proof of cannibalization?

Source grain is page-query, model grain is an unordered page pair, and output is a discovered
pair archetype plus review priority.

In [1]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
con.sql(f"""SELECT COUNT(*) candidate_pairs,
COUNT(DISTINCT client_hash_id) represented_clients,
MEDIAN(shared_query_count) median_shared_queries,
MEDIAN(weighted_query_overlap) median_overlap FROM {R}""").df()

,candidate_pairs,represented_clients,median_shared_queries,median_overlap
0,362562,43,3.0,0.016947


In [2]:
from pathlib import Path
import duckdb, numpy as np, pandas as pd
def find_root(p=Path.cwd()):
    for x in [p,*p.parents]:
        if (x/"skills"/"README.md").exists(): return x
    raise FileNotFoundError("Run inside repository")
ROOT=find_root()
PAIR_PATH=ROOT/"work"/"outputs"/"page_pair_features.parquet"
assert PAIR_PATH.exists() and PAIR_PATH.stat().st_size>0, "Run work/scripts/build_pair_features.py"
con=duckdb.connect()
R=f"read_parquet('{PAIR_PATH.as_posix()}')"
con.sql(f"""SELECT
SUM((growth_a>0 AND growth_b>0)::INT) shared_growth,
SUM((growth_a<0 AND growth_b<0)::INT) shared_decline,
SUM((growth_a*growth_b<0)::INT) opposite_movement,
SUM((growth_a IS NULL OR growth_b IS NULL)::INT) insufficient_history
FROM {R}""").df()

,shared_growth,shared_decline,opposite_movement,insufficient_history
0,45251.0,178522.0,122737.0,12582.0


## 4. Claim boundary

We can report measured overlap, balance, and movement and say a pair is consistent with a review
opportunity. We cannot confirm semantic equivalence, causation, or that merging improves ranking.
Every output is decision support.